In [ ]:
import pandas as pd
import json
from tqdm import tqdm
from explain import *
from pathlib import Path
from collections import Counter, defaultdict
import seaborn as sns

In [ ]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

TargetMaskedToken = 'man' # the token to be masked in the target sentence
try:
  import google.colab
  originalFolder = '.' # change to '.' when working in colab
  dataPath = '.' # change to '.' when working in colab 
  processedFolder = '.' # change '.' when working in colab
except:
  originalFolder = 'masking_data' # change to '.' when working in colab
  dataPath = 'input_data' # change to '.' when working in colab 
  processedFolder = 'gradient_data' # change '.' when working in colab

predCol = "pred_bert_1760_1900"
resultType = 'pred_kw_filtered' # pred | pred_kw_filter

print(f"This analysis focuses on '{TargetMaskedToken}'.")

In [ ]:
files = list(Path("/Volumes/work/Text-Machine").glob("blmicrosoft_final_m*n.jsonl"))
year2count = Counter()
for file in files:
    print(f"Processing {file.name}...")
    chunk_iter = pd.read_json(file, lines=True, chunksize=10_000)
    for chunk in tqdm(chunk_iter, desc=f"Processing {file.name}"):
        year2count = year2count + Counter(chunk['date'].values)
        

In [ ]:
df_year_count_1

In [ ]:
df_year_count =pd.DataFrame(list(year2count.items()), columns=['Year', 'Count']).set_index('Year')
df_year_count.to_csv(f"{processedFolder}/year_count_{collection}.csv")
df_year_count_1 = pd.read_csv(f"{processedFolder}/year_count_{collection}.csv")
df_year_count['decade'] = (df_year_count.index // 10) * 10
df_decade = df_year_count.groupby('decade').sum()#.set_index('decade').sort_index()
df_decade

In [ ]:
# load the original sentences with the predicted tokens
df_sent = pd.read_csv(f'{dataPath}/{collection}_{TargetMaskedToken}{genre_suffix}_{resultType}.tsv', index_col=0, sep='\t').reset_index(drop=True)
print(f'We have {df_sent.shape[0]} sentences that produced human predictions for the target token {TargetMaskedToken} in the {collection} collection.')
df_ig = pd.read_csv(f'{processedFolder}/results_{collection}_{TargetMaskedToken}_{resultType}_processed.csv', index_col=0 )
print(f'We have {df_ig.shape[0]} explanations for the target token {TargetMaskedToken} in the {collection} collection.')


In [ ]:
df_sent['decade'] = (df_sent['date'] // 10) * 10

In [ ]:
machine_words = ['machine','machines','machinery']

In [ ]:
for colName in ['pred_bert_contemporary', 'pred_bert_1760_1900']:
    df_sent[f'{colName}_machine'] = df_sent[colName].apply(
        lambda x: {w:s for w, s in dict(eval(x)).items() if w in machine_words})


In [ ]:

df_sent['machine_prediction'] = df_sent['pred_bert_1760_1900'].apply(lambda x: bool({w:s for w, s in dict(eval(x)).items() if w in machine_words}))
df_sent['machine_prediction_max'] = df_sent['pred_bert_1760_1900'].apply(lambda x: max([s for w, s in dict(eval(x)).items() if w in machine_words]+[0]))

In [ ]:
# from matplotlib import rcParams

# # figure size in inches
# rcParams['figure.figsize'] = 5.27,5.27
# thresholds = [0.1, 0.25, 0.5]
# for threshold in thresholds:
#     df_sent[f'max_machine_value_{threshold}'] = df_sent['pred_bert_1760_1900_machine'].apply(lambda x: max(list(x.values())+[.0]) >= threshold)
#     sns.lineplot(y=f'max_human_value_{threshold}',x='decade',data=df_sent[df_sent['date'].between(1800,1899)], alpha=0.9)

In [ ]:
threshold = .01
df_sent[f'max_machine_value_{threshold}'] = df_sent['pred_bert_1760_1900_machine'].apply(lambda x: max(list(x.values())+[.0]) >= threshold)
    

In [ ]:
df_sent.groupby('decade')[f'max_machine_value_{threshold}'].sum()

In [ ]:
(df_sent.groupby('decade')[f'max_machine_value_{threshold}'].sum() / df_decade['Count']).plot()

In [ ]:
import seaborn as sns
sns.scatterplot(x='date',y='machine_prediction_max',data=df_sent[df_sent['date'].between(1800,1899)], alpha=0.5)

In [ ]:
import seaborn as sns
sns.lineplot(x='decade',y='machine_prediction_max',data=df_sent[df_sent['date'].between(1800,1899)], alpha=0.5)

In [ ]:
df_sent

In [ ]:
pd.set_option('display.max_colwidth', 200)
df_sent.sort_values(by='machine_prediction_max', ascending=False).head(20)['currentSentence']

In [ ]:
# Create row order within each id and Target
df_ig["row_idx"] = df_ig.groupby(["id", "Target"]).cumcount()

# Extract man/men scores
man_scores = (
    df_ig[df_ig["Target"].isin(["man", "men"])]
    [["id", "row_idx", "Score_normalized"]]
    .rename(columns={"Score_normalized": "man_score"})
)

# Match each row with the corresponding machine row
df_ig = df_ig.merge(
    man_scores,
    on=["id", "row_idx"],
    how="left"
)

# Subtract
df_ig["diff"] = df_ig["Score_normalized"] - df_ig["man_score"]

# Optional: remove helper column
df_ig.drop(columns=["row_idx", "man_score"], inplace=True)

In [ ]:
threshold = 0.01

targetTokens = ['machine','machines'] # we look at the predictions for all the non machine words
df_comparisonConcept = df_ig[
    (df_ig['Target'].isin(targetTokens)) & # we exclude the target token itself, as we are interested in other tokens that are predictive of the contrastive concept
    (df_ig['Target_score'].between(threshold, 1.0))
                ].groupby('Token').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_diff=('diff', 'mean'), avg_score=('Score_normalized', 'mean')
                    ).reset_index()


In [ ]:
# please note that this repeats sentences, this acros all sentences with all the filtered keywords
min_count = 5
df_result = df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_score', ascending=False)
df_result.head(10)

In [ ]:
id =  7744#1637
sort_value = 'diff' #'Score_normalized' | 'diff'

comparisonTokens = ['machine','machines','machinery']
identifiers_ranked = df_ig.loc[df_ig['id'].isin(list(df_result.loc[id].identifiers)) \
                             & (df_ig['Token']==df_result.loc[id].Token) \
                             & df_ig['Target'].isin(comparisonTokens)
                             
                             ].sort_values(by=sort_value, ascending=False)['id'].unique()
sentences_ranked = df_sent.iloc[list(identifiers_ranked)].head(20)
sentences_ranked.currentSentence